In [0]:
from common.SilverPipelineClass import SilverPipeline
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

In [0]:
class MarketingSilverPipeline(SilverPipeline):
    def __init__(self):
        super().__init__(dominio='mkt')

    def transform(self, df:DataFrame, 
                  date_cols: list = None,
                  int_cols: list = None,
                  decimal_cols: list = None) -> 'pyspark.sql.DataFrame':
        # Transformação da classe pai
        df = super().transform(df)

        # Excluindo linhas com colunas nulas
        df = df.dropna()

        # === TRATAMENTO DE DATAS ===
        formats = ["yyyy-MM-dd", "dd-MM-yyyy", "dd/MM/yyyy"]

        if date_cols:
            for col in date_cols:
                df = df.withColumn(
                    f'{col}_limpa',
                    F.coalesce(*[F.try_to_date(F.col(f'{col}'), fmt) for fmt in formats])
                )

                df = df.filter(F.col(f'{col}_limpa').isNotNull())

                df = df \
                    .drop(f'{col}') \
                    .withColumnRenamed(
                        f'{col}_limpa', f'{col}'
                    )
            
        # === TRATAMENTO DE INTEIROS ===
        if int_cols:
            for col in int_cols:
                # O regexp_replace remove caracteres inválidos
                # O cast transforma valores inválidos (como letras) em null automaticamente
                df = df.withColumn(
                    f'{col}_limpa', 
                    F.regexp_replace(F.col(col).cast("string"), r"[^0-9]", "").cast("int")
                )
                
                # Filtra para remover linhas onde o código falhou no cast
                df = df.filter(F.col(f'{col}_limpa').isNotNull())
                
                df = df.drop(col).withColumnRenamed(f'{col}_limpa', col)
        
        # === TRATAMENTO DE DECIMAIS ===
        if decimal_cols:
            for col in decimal_cols:
                # O regexp_replace remove caracteres inválidos mas mantém pontos e vírgulas
                df = df.withColumn(
                    f'{col}_limpa', 
                    F.regexp_replace(F.col(col).cast("string"), r"[^0-9,.]", "")
                )
                
                # 2. Agora que sobrou só o número limpo, trocamos a vírgula pelo ponto decimal do Spark
                df = df.withColumn(
                    f'{col}_limpa', 
                    F.regexp_replace(F.col(f'{col}_limpa'), ",", ".").cast("decimal(18,2)")
                )
                
                # Filtra para remover linhas onde o código falhou no cast
                df = df.filter(F.col(f'{col}_limpa').isNotNull())
                df = df.drop(col).withColumnRenamed(f'{col}_limpa', col)
            
        return df
        
        
    

In [0]:
class ClienteMktPipeline(MarketingSilverPipeline):
    def __init__(self):
        super().__init__()

    def transform(self, df:DataFrame) -> 'pyspark.sql.DataFrame':
        print('Iniciando tratamento de cliente...')
        date_cols = ['data_cadastro']

        df = super().transform(df, date_cols)

        # Nomes são transformados em minúsculo e espaços em branco são removidos
        df = df \
            .withColumn('nome', F.trim(F.lower(F.col('nome')))) \
            .withColumn('cidade', F.trim(F.lower(F.col('cidade')))) \
            .withColumn('origem', F.trim(F.lower(F.col('origem'))))

        # === TRATAMENTO DE EMAILS ===
        # 1. Limpar
        df = df \
            .withColumn("email_processado", F.lower(F.trim(F.col('email'))))
        # 2. Validar
        regex_email = r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"
        df = df \
            .withColumn('email_valido', F.col('email_processado').rlike(regex_email))
        # 3. Tratamento final
        df = df \
            .withColumn('email', F.when(F.col('email_valido') == True, F.col('email_processado'))
                        .otherwise(None))
        # 4. Filtro    
        df = df.filter(F.col('email').isNotNull())
        # 5. Final
        df = df \
            .drop('email_processado', 'email_valido')

        # === TRATAMENTO DE TELEFONE ===
        # 1. Limpar
        df = df \
            .withColumn('telefone_limpo', F.regexp_replace(F.col('telefone'), r'[\(\)\s-]', '')) \
            .withColumn('telefone_valido', 
                        F.when(F.length(F.col('telefone_limpo')).between(10, 11), F.col('telefone_limpo'))
                        .otherwise(None))
        # 2. Retirar nulos
        df = df.filter(F.col('telefone_valido').isNotNull())
        # 3. Tratamento final
        df = df \
            .withColumn('telefone', F.col('telefone_valido')) \
            .drop('telefone_limpo', 'telefone_valido')

        # === TRATAMENTO DE ORIGEM ===
        df = df \
        .withColumn('origem',
                    F.when(F.col('origem').isin('facebook', 'face'), 'facebook')
                    .when(F.col('origem').isin('instagram', 'insta'), 'instagram')
                    .when(F.col('origem').isin('999'), None)
                    .otherwise(F.col('origem'))
        )
        df = df.filter(F.col('origem').isNotNull())
        
        print('Tratamento finalizado com sucesso!')
        return df

In [0]:
cliente = ClienteMktPipeline()
cliente.run('cliente')